# Evaluating a Classification Prompt

A prompt is tuned like a machine learning model: you measure it on a **test set** whose right answers you know, rather than judging it on two or three tries.

Here, an LLM sorts IT support tickets into five categories. We compare several versions of the prompt on 40 hand-labeled tickets.

The model: [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B), a small open model. A GPU is recommended (`Runtime > Change runtime type`). Without a GPU, the notebook loads the 0.6B version instead: lighter, but much less accurate.

In [1]:
import warnings

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SIZE = "1.7B" if device == "cuda" else "0.6B"  # 0.6B: light enough for a CPU

if device == "cpu":
    warnings.warn("No GPU found: the notebook loads the 0.6B model, which is much less accurate. "
                  "For the 1.7B model, pick a GPU (Runtime > Change runtime type), then rerun the notebook.")

LLM = f"Qwen/Qwen3-{MODEL_SIZE}"
DTYPE = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(LLM, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map=device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## The test set

40 tickets, 8 per category, with the category support would have chosen. Some are ambiguous on purpose: a phishing email is about email, but belongs to security; a VPN password is about the network, but belongs to access.

In [2]:
CATEGORIES = ["access", "network", "email", "workstation", "security"]

TICKETS = [
    ("My account is locked after several wrong passwords.", "access"),
    ("I don't have the rights to open the Finance shared folder.", "access"),
    ("My VPN password has expired, how do I change it?", "access"),
    ("New hire on Monday: please create their account and mailbox.", "access"),
    ("I no longer receive the two-factor authentication codes on my phone.", "access"),
    ("Can I get write access to the team wiki?", "access"),
    ("I forgot my Windows password.", "access"),
    ("Outlook asks for my password again every time it starts.", "access"),
    ("The VPN disconnects every ten minutes.", "network"),
    ("No wifi in meeting room B.", "network"),
    ("The internet has been very slow on the whole floor since this morning.", "network"),
    ("The file server can't be reached from the Lyon branch.", "network"),
    ("The VPN won't connect: \"gateway unreachable\" error.", "network"),
    ("Video calls keep dropping from the Nantes site.", "network"),
    ("The network socket in my office no longer works.", "network"),
    ("The intranet site is down, for everyone.", "network"),
    ("I haven't received any emails since last night.", "email"),
    ("My mailbox is full, I can't send messages anymore.", "email"),
    ("How do I share my calendar with my assistant?", "email"),
    ("A customer says my emails end up in their spam folder.", "email"),
    ("I'd like a mailing list for the project team.", "email"),
    ("A 30 MB attachment is rejected when sending.", "email"),
    ("My out-of-office replies aren't sent.", "email"),
    ("Meeting invitations arrive with a one-hour offset.", "email"),
    ("My screen stays black at startup.", "workstation"),
    ("The 3rd-floor printer prints blank pages.", "workstation"),
    ("I need Excel on my new laptop.", "workstation"),
    ("My computer has been very slow since the last update.", "workstation"),
    ("Several keys on my laptop keyboard don't respond anymore.", "workstation"),
    ("My laptop battery only lasts an hour now.", "workstation"),
    ("The accounting software crashes when it opens.", "workstation"),
    ("My headset isn't recognized by the computer.", "workstation"),
    ("An email from \"IT\" asks me to confirm my password through a link.", "security"),
    ("I clicked a link in an email and a page asked for my credentials.", "security"),
    ("The antivirus reports an infected file on my USB stick.", "security"),
    ("I lost my work laptop on the train.", "security"),
    ("Pop-up ads open by themselves on my computer.", "security"),
    ("Someone logged into my account from abroad last night.", "security"),
    ("A supplier asks by email to change their bank details before the next payment.", "security"),
    ("I sent a customer file to an outside address by mistake.", "security"),
]
test_set = pd.DataFrame(TICKETS, columns=["ticket", "expected"])
test_set.expected.value_counts()

,count
expected,
access,8
network,8
email,8
workstation,8
security,8


## Measuring a prompt

The `evaluate` function sends each ticket to the model with the prompt (`{ticket}` is replaced by the ticket's text), reads the first category named in the answer, and computes the share of right answers. An answer that names no category counts as an error, shown as `?`.

In [3]:
def ask(prompts: list[str], batch_size: int = 8) -> list[str]:
    answers = []
    for start in range(0, len(prompts), batch_size):
        messages = [[{"role": "user", "content": p}] for p in prompts[start:start + batch_size]]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                               padding=True, return_tensors="pt", return_dict=True).to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=30, do_sample=False, temperature=None, top_p=None,
                                    top_k=None, pad_token_id=tokenizer.eos_token_id)
        answers += tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return answers


def category(answer: str) -> str:
    answer = answer.lower()
    found = [(answer.find(c), c) for c in CATEGORIES if c in answer]
    return min(found)[1] if found else "?"


def evaluate(prompt: str) -> pd.DataFrame:
    answers = ask([prompt.replace("{ticket}", t) for t in test_set.ticket])
    results = test_set.assign(answer=answers, predicted=[category(a) for a in answers])
    accuracy = (results.predicted == results.expected).mean()
    print(f"Accuracy: {accuracy:.0%} ({(results.predicted == results.expected).sum()}/{len(results)})")
    return results


def errors(results: pd.DataFrame) -> pd.DataFrame:
    return results[results.predicted != results.expected][["ticket", "expected", "predicted"]]

## Version 1: the naive prompt

In [4]:
PROMPT_V1 = """Put this ticket in a category: access, network, email, workstation or security.

Ticket: {ticket}"""

v1 = evaluate(PROMPT_V1)
v1.answer.head()

Accuracy: 35% (14/40)


,answer
0,The appropriate category for this ticket is: *...
1,The appropriate category for this ticket is: *...
2,The appropriate category for this ticket is: *...
3,The appropriate category for this ticket is: *...
4,The appropriate category for this ticket is: *...


The model answers with sentences of varying length: easy for a human to read, less so for a program. Which mistakes does it make?

In [5]:
errors(v1)

,ticket,expected,predicted
0,My account is locked after several wrong passw...,access,security
1,I don't have the rights to open the Finance sh...,access,security
2,"My VPN password has expired, how do I change it?",access,security
3,New hire on Monday: please create their accoun...,access,workstation
4,I no longer receive the two-factor authenticat...,access,security
5,Can I get write access to the team wiki?,access,email
6,I forgot my Windows password.,access,security
7,Outlook asks for my password again every time ...,access,security
20,I'd like a mailing list for the project team.,email,network
21,A 30 MB attachment is rejected when sending.,email,network


## Version 2: defining the categories and the format

The model doesn't know our conventions: where does a VPN password go, or a phishing email? We define each category, with the priority rules, and enforce the answer's format.

In [6]:
PROMPT_V2 = """You sort IT support tickets. Categories:
- access: accounts, passwords (including the VPN's or the mailbox's), rights, two-factor authentication
- network: VPN, wifi, internet connection, slow or dropped network, unreachable internal sites
- email: emails, calendar, mailing lists, attachments
- workstation: the workstation's hardware and software, printers
- security: suspicious or fraudulent email, virus, lost or stolen equipment, suspicious login, data leak

Rule: anything about a threat or a security incident belongs to "security", even if it goes through an email or an account.

Answer with the category name only, in lowercase.

Ticket: {ticket}"""

v2 = evaluate(PROMPT_V2)
errors(v2)

Accuracy: 65% (26/40)


,ticket,expected,predicted
0,My account is locked after several wrong passw...,access,security
1,I don't have the rights to open the Finance sh...,access,security
2,"My VPN password has expired, how do I change it?",access,security
4,I no longer receive the two-factor authenticat...,access,security
5,Can I get write access to the team wiki?,access,security
6,I forgot my Windows password.,access,security
7,Outlook asks for my password again every time ...,access,security
16,I haven't received any emails since last night.,email,security
17,"My mailbox is full, I can't send messages anym...",email,security
18,How do I share my calendar with my assistant?,email,workstation


## Version 3: adding examples

We add a few solved examples, picked among the hard cases, but **not in the test set**: otherwise we would measure the model's memory, not its ability to generalize.

In [7]:
EXAMPLES = """Examples:
Ticket: My VPN access is locked after three wrong passwords. → access
Ticket: Screen sharing freezes during remote meetings with the branch. → network
Ticket: I can't find my old archived emails. → email
Ticket: My wireless mouse stopped working. → workstation
Ticket: A text message in the name of the company's bank asks for my credentials. → security

"""

PROMPT_V3 = PROMPT_V2.replace("Ticket: {ticket}", EXAMPLES + "Ticket: {ticket}")

v3 = evaluate(PROMPT_V3)
errors(v3)

Accuracy: 72% (29/40)


,ticket,expected,predicted
0,My account is locked after several wrong passw...,access,security
1,I don't have the rights to open the Finance sh...,access,security
2,"My VPN password has expired, how do I change it?",access,security
3,New hire on Monday: please create their accoun...,access,security
4,I no longer receive the two-factor authenticat...,access,security
5,Can I get write access to the team wiki?,access,security
6,I forgot my Windows password.,access,security
7,Outlook asks for my password again every time ...,access,security
15,"The intranet site is down, for everyone.",network,security
19,A customer says my emails end up in their spam...,email,security


## Comparing the versions

Right answers per category, for each version:

In [8]:
pd.DataFrame({
    version: (results.predicted == results.expected).groupby(results.expected).sum()
    for version, results in {"v1": v1, "v2": v2, "v3": v3}.items()
})

,v1,v2,v3
expected,,,
access,0,1,0
email,4,1,6
network,8,8,7
security,2,8,8
workstation,0,8,8


Each version fixes some errors and may create others: v2's definitions bring security from 2 to 8 right answers out of 8, and workstation from 0 to 8, but email drops to 1 and access stays at 1: the model files nearly every account problem under "security", as if a password were a security matter. v3's examples fix email, not access, which drops to 0: its 72% hides a category the model never gets right. Hence measuring category by category, not only overall.

The best version's confusion matrix: the expected category in rows, the predicted one in columns.

In [9]:
pd.crosstab(v3.expected, v3.predicted)

predicted,email,network,security,workstation
expected,,,,
access,0,0,8,0
email,6,1,1,0
network,0,7,1,0
security,0,0,8,0
workstation,0,0,0,8


With 40 tickets, one ticket is worth 2.5 points: a 5-point gap between two versions may be chance. A real test set rather counts a hundred cases, taken from real tickets.

## Your turn

Write your own version of the prompt, with `{ticket}` where the ticket goes, and measure it. For instance, try to fix v3's weakest category.

In [10]:
MY_PROMPT = """Classify this IT support ticket: access, network, email, workstation or security.
Answer with a single word.

Ticket: {ticket}"""

mine = evaluate(MY_PROMPT)
errors(mine)

Accuracy: 48% (19/40)


,ticket,expected,predicted
1,I don't have the rights to open the Finance sh...,access,security
2,"My VPN password has expired, how do I change it?",access,network
4,I no longer receive the two-factor authenticat...,access,security
6,I forgot my Windows password.,access,security
7,Outlook asks for my password again every time ...,access,?
18,How do I share my calendar with my assistant?,email,?
21,A 30 MB attachment is rejected when sending.,email,access
22,My out-of-office replies aren't sent.,email,network
23,Meeting invitations arrive with a one-hour off...,email,network
24,My screen stays black at startup.,workstation,access


## Key takeaways

- A prompt is judged on a labeled test set, not on a few tries
- Defining the categories and our conventions, enforcing the format, giving examples: each addition can be measured
- The prompt's examples must not come from the test set
- This test set is replayed on every change of prompt or model